# Thursday, hands on: the customer table, refreshable in one run
#
# > "One table, one row per customer, refreshed on Monday."
#
# Replace every `__TODO__`. The last step has no check, because no check can grade a judgment.

In [ ]:
import pathlib
import sys

import pandas as pd

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

eng = kit.engine()
orders = pd.read_sql("SELECT * FROM orders", eng)
customers = pd.read_sql("SELECT * FROM customers", eng)
exposure = pd.read_sql("SELECT * FROM campaign_exposure", eng)
kit.flow(["read", "group", "merge", "reshape", "choose a tool"], lit=[0], title="Where you are")

## 1. Recency, frequency, monetary
#
# One row per customer carrying their order count, their total spend and their latest order date.
# Use named aggregations rather than the short form, so each output column says where it came
# from.

In [ ]:
AS_OF = pd.Timestamp("2026-09-30")
table = __TODO1__
kit.check("one row per customer who ordered", table["customer_id"].is_unique)
kit.check("three measures plus the key", table.shape[1] == 4, f"{table.shape[1]} columns")

## 2. Recency in days
#
# Days since the last order, counted from `AS_OF` rather than from today. State the date, and the
# table means the same thing whenever it runs.

In [ ]:
table["recency_days"] = __TODO2__
kit.check("recency is never negative", table["recency_days"].min() >= 0,
          f"min {table['recency_days'].min()}")

## 3. Meet the merge failure on purpose
#
# Merge the exposure feed on `customer_id` with `validate="one_to_one"`. Read what comes back.

In [ ]:
try:
    __TODO3__
    print("no error: check that validate= is set")
except Exception as e:
    print(type(e).__module__ + "." + type(e).__name__)
    print(str(e))
kit.tree({"label": "duplicate keys", "branches": [
    ("drop them", {"label": "picks a row arbitrarily"}),
    ("aggregate first", {"label": "keeps both facts"}),
    ("ask the owner", {"label": "fixes it upstream"})]},
    taken=["aggregate first"], title="Three ways out")

## 4. Fix it, then merge cleanly
#
# Collapse the exposure feed to one row per customer first, then merge with validation on. The
# row count must not move.

In [ ]:
seen = __TODO4__
before = len(table)
table = table.merge(seen, on="customer_id", how="left", validate="one_to_one")
table["exposed"] = table["exposed"].fillna(0).astype(int)
kit.check("the validated merge kept the row count", len(table) == before, f"{len(table)}")

## 5. Segment on
#
# Bring segment and city across from `customers`. Choose the `validate=` value that says what you
# believe about the two tables, and be ready to defend it.

In [ ]:
table = __TODO5__
kit.check("no customer lost a segment", table["segment"].notna().all())
kit.ladder(["orders", "grouped", "exposure merged", "segment merged"], lit=[3],
           title="The table, built")

## 6. Reshape for comparison
#
# Q2 monthly spend per customer, months as columns. Then say aloud what one row of your result
# means before you look at any number in it.

In [ ]:
wide = __TODO6__
kit.check("three months across", wide.shape[1] == 3, f"{wide.shape[1]} columns")
kit.matrix(["long", "wide"], ["one row is", "easy question"],
           [["a customer-month", "how did this move"],
            ["a customer", "how do these compare"]], title="Same numbers, two shapes")

## 7. The tool note
#
# Four or five sentences answering the senior analyst: which of plain Python, SQL and pandas
# answers which of Marketing's and Finance's asks, and why.
#
# Include the one you would refuse and the reason. A note without a refusal has not made a choice.
# No check can grade this, which is why it is the deliverable.

In [ ]:
kit.decision_ladder(["name the three tools", "say which is fastest",
                     "say which owns which number, and why",
                     "say which you would refuse, and why"], cut_at=3,
                    title="What the senior analyst is actually asking for")
kit.check_summary()